In [ ]:
import torch
import torchvision.models as models
from torch.profiler import profile, ProfilerActivity, record_function

In [ ]:
model = models.resnet18()
inputs = torch.randn(5,3, 224, 224)

In [ ]:
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    with record_function("model_inference"):
        model(inputs)

In [ ]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

In [ ]:
prof.export_chrome_trace("trace.json") # view in perfetto UI

In [ ]:
device="cpu"
activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]
sort_by_keyword = "self_" + device + "_time_total"

with profile(
    activities=activities,
    with_stack=True,
    experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
) as prof:
    model(inputs)

# Print aggregated stats
print(prof.key_averages(group_by_stack_n=5).table(sort_by=sort_by_keyword, row_limit=20))
